In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.utils.features import split_features

In [2]:
DATA_DIR = Path("../datasets/final/ml")

DATASET_NAMES = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]

RANDOM_STATE = 1
N_SPLITS = 5

In [3]:
def build_logistic_pipeline(numerical_features, categorical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )),
        ]
    )

In [4]:
def summarize_coefficients(estimators):
    coefficients_by_fold = []

    for fold, estimator in enumerate(estimators, start=1):
        preprocessor = estimator.named_steps["preprocessor"]
        model = estimator.named_steps["model"]

        feature_names = [
            name.removeprefix("numerical__").removeprefix("categorical__")
            for name in preprocessor.get_feature_names_out()
        ]

        coefficients_by_fold.append(pd.DataFrame({
            "fold": fold,
            "feature": feature_names,
            "coefficient": model.coef_[0],
        }))

    coefficients = pd.concat(coefficients_by_fold, ignore_index=True)
    summary = (
        coefficients
        .groupby("feature")["coefficient"]
        .agg(["mean", "std"])
        .rename(columns={
            "mean": "coefficient_mean",
            "std": "coefficient_std",
        })
        .reset_index()
    )

    summary["abs_coefficient_mean"] = summary["coefficient_mean"].abs()

    return summary.sort_values("abs_coefficient_mean", ascending=False)


def evaluate_training_dataset(dataset_name):
    df = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_train.parquet"
    )

    numerical_features, categorical_features = split_features(df)
    feature_columns = numerical_features + categorical_features

    features = df[feature_columns]
    target = df["label_binary"].eq("ATTACK").astype(int)

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scores = cross_validate(
        build_logistic_pipeline(
            numerical_features,
            categorical_features,
        ),
        features,
        target,
        cv=cross_validation,
        scoring={
            "precision_attack": make_scorer(
                precision_score,
                pos_label=1,
                zero_division=0,
            ),
            "recall_attack": make_scorer(
                recall_score,
                pos_label=1,
                zero_division=0,
            ),
            "f1_attack": make_scorer(f1_score, pos_label=1),
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
        },
        n_jobs=1,
        return_estimator=True,
    )

    metrics = {
        "dataset": dataset_name,
        "numerical_features": len(numerical_features),
        "categorical_features": len(categorical_features),
        "precision_attack_mean": scores["test_precision_attack"].mean(),
        "precision_attack_std": scores["test_precision_attack"].std(),
        "recall_attack_mean": scores["test_recall_attack"].mean(),
        "recall_attack_std": scores["test_recall_attack"].std(),
        "f1_attack_mean": scores["test_f1_attack"].mean(),
        "f1_attack_std": scores["test_f1_attack"].std(),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": scores["test_balanced_accuracy"].std(),
    }

    return metrics, summarize_coefficients(scores["estimator"])

In [5]:
results = []
coefficient_summaries = {}

for dataset_name in DATASET_NAMES:
    metrics, coefficient_summary = evaluate_training_dataset(dataset_name)
    results.append(metrics)
    coefficient_summaries[dataset_name] = coefficient_summary

results_df = pd.DataFrame(results)

display(results_df)

for dataset_name in DATASET_NAMES:
    print(f"\nCoeficientes - {dataset_name}")
    display(coefficient_summaries[dataset_name])

,dataset,numerical_features,categorical_features,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,cicids2017,14,1,0.884812,0.003477,0.903881,0.001988,0.894238,0.001603,0.957238,0.000721,0.937229,0.000864
1,unsw_nb15,14,1,0.933971,0.002666,0.814684,0.003137,0.870254,0.002100,0.951416,0.000755,0.900142,0.001561
2,iot23,14,1,0.995249,0.000542,0.991719,0.000581,0.993481,0.000375,0.997397,0.000150,0.995268,0.000291



Coeficientes - cicids2017


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
14,protocol_17,-5.403285,0.276139,5.403285
12,bidirectional_stddev_ps,4.572518,0.102416,4.572518
9,bidirectional_psh_packets,-4.323078,1.006442,4.323078
5,bidirectional_max_ps,-1.914437,0.084277,1.914437
15,protocol_6,1.781002,0.122083,1.781002
10,bidirectional_rst_packets,1.404435,0.005954,1.404435
3,bidirectional_fin_packets,-1.335744,0.022588,1.335744
7,bidirectional_mean_ps,-1.178940,0.031025,1.178940
4,bidirectional_max_piat_ms,-0.785020,0.016112,0.785020
13,bidirectional_syn_packets,0.719555,0.017478,0.719555



Coeficientes - unsw_nb15


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
0,bidirectional_ack_packets,51.925409,1.327665,51.925409
9,bidirectional_psh_packets,-44.548629,0.582056,44.548629
1,bidirectional_bytes,-9.331505,1.252018,9.331505
8,bidirectional_packets,6.819588,1.206438,6.819588
12,bidirectional_stddev_ps,-4.674984,0.093284,4.674984
5,bidirectional_max_ps,3.468287,0.085295,3.468287
16,protocol_89,-2.985556,0.144111,2.985556
6,bidirectional_mean_piat_ms,0.966197,0.402046,0.966197
11,bidirectional_stddev_piat_ms,-0.905626,0.429156,0.905626
3,bidirectional_fin_packets,0.784662,0.066501,0.784662



Coeficientes - iot23


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
6,bidirectional_mean_piat_ms,-15.841133,0.250719,15.841133
14,protocol_17,-8.947129,0.659216,8.947129
2,bidirectional_duration_ms,5.979468,0.226047,5.979468
9,bidirectional_psh_packets,5.905994,0.576817,5.905994
4,bidirectional_max_piat_ms,4.382291,0.589770,4.382291
15,protocol_6,4.040690,0.282687,4.040690
3,bidirectional_fin_packets,3.819947,0.775011,3.819947
0,bidirectional_ack_packets,-3.286300,0.358996,3.286300
8,bidirectional_packets,-2.545866,0.216069,2.545866
5,bidirectional_max_ps,2.281348,0.455180,2.281348


## Validação cross-dataset de protocol

In [6]:
def evaluate_protocol_cross_dataset():
    results = []

    for source_name in DATASET_NAMES:
        source_df = pd.read_parquet(
            DATA_DIR / f"{source_name}_train.parquet"
        )

        numerical_features, categorical_features = split_features(source_df)

        for use_protocol in [True, False]:
            current_categorical = categorical_features.copy()

            if not use_protocol:
                current_categorical.remove("protocol")

            feature_columns = numerical_features + current_categorical
            source_target = (
                source_df["label_binary"].eq("ATTACK").astype(int)
            )

            pipeline = build_logistic_pipeline(
                numerical_features,
                current_categorical,
            )
            pipeline.fit(source_df[feature_columns], source_target)

            for target_name in DATASET_NAMES:
                if target_name == source_name:
                    continue

                target_df = pd.read_parquet(
                    DATA_DIR / f"{target_name}_train.parquet",
                    columns=feature_columns + ["label_binary"],
                )
                target = (
                    target_df["label_binary"].eq("ATTACK").astype(int)
                )
                predictions = pipeline.predict(target_df[feature_columns])

                results.append({
                    "source_dataset": source_name,
                    "validation_dataset": target_name,
                    "protocol": "included" if use_protocol else "excluded",
                    "precision_attack": precision_score(
                        target,
                        predictions,
                        zero_division=0,
                    ),
                    "recall_attack": recall_score(
                        target,
                        predictions,
                        zero_division=0,
                    ),
                    "f1_attack": f1_score(target, predictions),
                    "accuracy": accuracy_score(target, predictions),
                    "balanced_accuracy": balanced_accuracy_score(
                        target,
                        predictions,
                    ),
                })

    return pd.DataFrame(results)

In [7]:
protocol_results = evaluate_protocol_cross_dataset()
display(protocol_results)

metric_columns = [
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "accuracy",
    "balanced_accuracy",
]
index_columns = ["source_dataset", "validation_dataset"]

included = (
    protocol_results[protocol_results["protocol"] == "included"]
    .set_index(index_columns)
)
excluded = (
    protocol_results[protocol_results["protocol"] == "excluded"]
    .set_index(index_columns)
)

# diferença = sem protocol - com protocol
#lembrete: valores negativos indicam piora após a remoção
protocol_impact = excluded[metric_columns] - included[metric_columns]
protocol_impact.columns = [
    f"{column}_difference" for column in metric_columns
]

display(protocol_impact.reset_index())

,source_dataset,validation_dataset,protocol,precision_attack,recall_attack,f1_attack,accuracy,balanced_accuracy
0,cicids2017,unsw_nb15,included,0.089011,0.001720,0.003374,0.796824,0.498660
1,cicids2017,iot23,included,0.028212,0.103232,0.044314,0.109478,0.107136
2,cicids2017,unsw_nb15,excluded,0.084926,0.001699,0.003331,0.796679,0.498562
3,cicids2017,iot23,excluded,0.028197,0.103062,0.044280,0.110221,0.107536
4,unsw_nb15,cicids2017,included,0.626469,0.447153,0.521836,0.836108,0.690250
5,unsw_nb15,iot23,included,0.004804,0.017283,0.007518,0.087324,0.061059
6,unsw_nb15,cicids2017,excluded,0.606397,0.456453,0.520848,0.832035,0.691192
7,unsw_nb15,iot23,excluded,0.003887,0.013971,0.006082,0.086793,0.059485
8,iot23,cicids2017,included,0.489441,0.448766,0.468222,0.796127,0.665867
9,iot23,unsw_nb15,included,0.252404,0.062975,0.100800,0.775290,0.508172


,source_dataset,validation_dataset,precision_attack_difference,recall_attack_difference,f1_attack_difference,accuracy_difference,balanced_accuracy_difference
0,cicids2017,unsw_nb15,-0.004085,-0.000021,-0.000044,-0.000144,-0.000098
1,cicids2017,iot23,-0.000015,-0.000170,-0.000034,0.000743,0.000401
2,unsw_nb15,cicids2017,-0.020071,0.009300,-0.000988,-0.004072,0.000942
3,unsw_nb15,iot23,-0.000916,-0.003312,-0.001435,-0.000531,-0.001574
4,iot23,cicids2017,0.137827,-0.100705,-0.020521,0.032120,-0.017689
5,iot23,unsw_nb15,-0.214913,-0.057455,-0.091177,-0.002531,-0.023127
